<a href="https://colab.research.google.com/github/vidhya2026/Duffl_Career_Page/blob/main/RM_NCDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Libraries

In [1]:
!pip install sentence-transformers transformers pdfplumber python-docx spacy flask pyngrok -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 29.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 34.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
#Imports & Model Loading

import os, re, tempfile
import pdfplumber
import spacy
import pandas as pd
from docx import Document
from datetime import datetime
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from flask import Flask, request, jsonify
from pyngrok import ngrok
from google.colab import files
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings("ignore")

print("Loading models...")
embedder   = SentenceTransformer('all-MiniLM-L6-v2')
ner_pipe   = pipeline("ner", model="dslim/bert-base-NER",    aggregation_strategy="simple")
ner_large  = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english", aggregation_strategy="simple")
nlp        = spacy.load("en_core_web_sm")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
print("Models loaded!")

Loading models...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Models loaded!


In [3]:
#  Text Extraction (PDF / DOCX / TXT)

def extract_text_pdf(path):
    full, left, right = "", "", ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            x0, y0, x1, y1 = page.bbox
            t = page.extract_text()
            if t: full += t + "\n"
            mid_x = x0 + (x1 - x0) * 0.42
            try:
                lt = page.crop((x0, y0, mid_x, y1)).extract_text()
                if lt: left += lt + "\n"
            except Exception: pass
            try:
                rt = page.crop((mid_x, y0, x1, y1)).extract_text()
                if rt: right += rt + "\n"
            except Exception: pass
    return full.strip(), left.strip(), right.strip()

def extract_text_docx(path):
    doc  = Document(path)
    text = "\n".join([p.text for p in doc.paragraphs if p.text.strip()])
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                text += "\n" + cell.text
    return text.strip(), "", ""

def extract_text(path):
    if path.lower().endswith(".pdf"):  return extract_text_pdf(path)
    if path.lower().endswith(".docx"): return extract_text_docx(path)
    if path.lower().endswith(".txt"):
        with open(path, "r", errors="ignore") as f: c = f.read()
        return c, "", ""
    return "", "", ""

In [4]:
# Constants & Blocklists


EDUCATION_KEYWORDS = [
    "university","college","institute","school","b.tech","m.tech",
    "b.e","m.e","bsc","msc","mba","bca","mca","bachelor","master",
    "degree","diploma","10th","12th","sslc","hsc","cbse","icse",
    "engineering college","polytechnic","iit","nit","bits","vit",
    "anna university","pune university","mumbai university",
    "graduation","post graduation","pg","ug","phd","doctorate",
]

TOOL_TECH_BLOCKLIST = {
    "balsamiq","figma","sketch","invision","zeplin","axure","adobe xd",
    "canva","miro","lucidchart","draw.io","whimsical","overflow",
    "framer","principle","marvel","proto.io","origami",
    "vscode","visual studio","eclipse","intellij","pycharm","sublime",
    "atom","notepad++","vim","emacs","webstorm","rider","xcode",
    "android studio","netbeans","codeblocks",
    "jira","confluence","trello","asana","notion","slack","teams",
    "basecamp","monday","clickup","github","gitlab","bitbucket",
    "jenkins","bamboo","circleci","travis",
    "react","angular","vue","django","flask","spring","laravel",
    "express","rails","fastapi","dotnet","asp.net","jquery",
    "bootstrap","tailwind","redux","graphql","rest","soap",
    "tensorflow","pytorch","keras","scikit","pandas","numpy",
    "matplotlib","seaborn","plotly","spark","hadoop","kafka",
    "mysql","postgresql","mongodb","sqlite","oracle","mssql",
    "redis","cassandra","dynamodb","firebase","supabase",
    "elasticsearch","neo4j","hbase",
    "aws","azure","gcp","docker","kubernetes","terraform",
    "ansible","puppet","chef","nginx","apache","linux","ubuntu",
    "centos","debian","windows server","vmware",
    "selenium","cypress","jest","mocha","junit","testng",
    "postman","soapui","appium","robotframework","cucumber",
    "loadrunner","jmeter","katalon",
    "tableau","power bi","qlik","looker","superset","metabase",
    "excel","google sheets","google analytics","mixpanel","segment",
    "agile","scrum","kanban","devops","ci/cd","microservices",
    "wordpress","shopify","salesforce","sap","oracle erp",
    "photoshop","illustrator","premiere","after effects",
    "unity","unreal","godot",
}

# City-only — NO country/state names
KNOWN_CITIES = {
    "bangalore","bengaluru","mumbai","delhi","new delhi","hyderabad",
    "chennai","pune","kolkata","ahmedabad","surat","jaipur","lucknow",
    "noida","gurgaon","gurugram","chandigarh","bhopal","indore","nagpur",
    "vizag","visakhapatnam","kochi","cochin","coimbatore","madurai",
    "trichy","tiruchirappalli","trivandrum","thiruvananthapuram",
    "thrissur","kozhikode","calicut","salem","erode","tirunelveli",
    "vellore","mysore","mysuru","hubli","belgaum","mangalore","udupi",
    "bhubaneswar","patna","ranchi","raipur","guwahati",
    "dehradun","shimla","srinagar","jammu",
    "jodhpur","udaipur","ajmer","kota","bikaner",
    "varanasi","prayagraj","kanpur","agra","meerut",
    "ghaziabad","faridabad","amritsar","ludhiana","jalandhar",
    "vadodara","rajkot","gandhinagar","anand",
    "navi mumbai","thane","aurangabad","nashik","kolhapur","solapur",
    "pondicherry","puducherry","ooty",
    "remote","hybrid","onsite",
    "new york","los angeles","chicago","houston","san francisco",
    "seattle","denver","boston","atlanta","miami","dallas","austin",
    "london","manchester","birmingham","toronto","vancouver","montreal",
    "sydney","melbourne","brisbane","perth",
    "dubai","abu dhabi","doha","singapore","kuala lumpur",
    "tokyo","osaka","seoul","taipei","hong kong","shanghai","beijing",
    "berlin","munich","paris","amsterdam","zurich","rome","madrid",
    "johannesburg","cape town","nairobi",
}

INVALID_WORDS = {
    "tmp","temp","naukri","indeed","linkedin","monster","shine","foundit",
    "glassdoor","hirist","instahyre","apna","freshersworld","timesjobs",
    "workindia","internshala","iimjobs","ui","ux","uiux","uxui","dev",
    "devops","qa","ba","pm","frontend","backend","fullstack","full","stack",
    "data","cloud","ml","ai","hr","it","ios","web","sde","swe","react",
    "angular","vue","java","python","dotnet","node","flutter","android",
    "php","ruby","golang","rust","aws","azure","gcp","docker","kubernetes",
    "resume","curriculum","vitae","profile","contact","summary","objective",
    "experience","education","skills","address","phone","email","mobile",
    "github","portfolio","work","professional","personal","details",
    "information","career","about","project","references","developer",
    "engineer","manager","analyst","designer","consultant","specialist",
    "lead","head","senior","junior","intern","fresher","candidate",
    "applicant","updated","new","final","cv","doc","copy","sample","test",
    "example","demo","blank","template","format","latest","hire","upload",
    "document","file","draft","version",
}

SPECIFIC_TECH_DOMAIN_MAP = [
    (re.compile(
        r'\b(qa\s+engineer|quality\s+assurance\s+engineer|quality\s+analyst|'
        r'automation\s+tester|manual\s+tester|software\s+tester|sdet|'
        r'test\s+engineer|testing\s+engineer|qa\s+lead|qa\s+analyst|'
        r'associate\s+qa|junior\s+(?:software\s+)?tester|'
        r'software\s+test(?:ing)?\s+intern(?:ship)?|'
        r'qa\s+intern(?:ship)?|test(?:ing)?\s+intern(?:ship)?|'
        r'software\s+(?:quality|testing)|quality\s+engineer)\b',
        re.IGNORECASE), "QA / Test Engineer"),
    (re.compile(r'\b(\.net|asp\.net|c#|csharp|wpf|xaml|blazor|entity\s+framework|ef\s+core)\b', re.IGNORECASE), ".NET Developer"),
    (re.compile(r'\b(java\s+developer|spring\s+boot|spring\s+mvc|hibernate|j2ee|javaee)\b', re.IGNORECASE), "Java Developer"),
    (re.compile(r'\b(python\s+developer|django\s+developer|fastapi\s+developer)\b', re.IGNORECASE), "Python Developer"),
    (re.compile(r'\b(react\s+developer|react\.js\s+developer|next\.js\s+developer)\b', re.IGNORECASE), "React Developer"),
    (re.compile(r'\b(angular\s+developer|angularjs\s+developer)\b', re.IGNORECASE), "Angular Developer"),
    (re.compile(r'\b(node\s+developer|node\.js\s+developer|nodejs\s+developer)\b', re.IGNORECASE), "Node.js Developer"),
    (re.compile(r'\b(flutter\s+developer|dart\s+developer)\b', re.IGNORECASE), "Flutter Developer"),
    (re.compile(r'\b(android\s+developer|kotlin\s+developer)\b', re.IGNORECASE), "Android Developer"),
    (re.compile(r'\b(ios\s+developer|swift\s+developer)\b', re.IGNORECASE), "iOS Developer"),
    (re.compile(r'\b(data\s+scientist|machine\s+learning\s+engineer|ml\s+engineer|ai\s+engineer)\b', re.IGNORECASE), "Data Scientist / ML Engineer"),
    (re.compile(r'\b(data\s+analyst|bi\s+analyst|business\s+intelligence\s+analyst)\b', re.IGNORECASE), "Data Analyst"),
    (re.compile(r'\b(data\s+engineer|etl\s+developer)\b', re.IGNORECASE), "Data Engineer"),
    (re.compile(r'\b(devops\s+engineer|cloud\s+engineer|site\s+reliability\s+engineer|sre|infrastructure\s+engineer|platform\s+engineer)\b', re.IGNORECASE), "DevOps / Cloud Engineer"),
    (re.compile(r'\b(ui[\s/]*ux\s+designer|product\s+designer|ux\s+designer|ui\s+designer|ux\s+researcher|interaction\s+designer|visual\s+designer)\b', re.IGNORECASE), "UI/UX Designer"),
    (re.compile(r'\b(full[\s\-]?stack\s+developer|fullstack\s+developer)\b', re.IGNORECASE), "Full Stack Developer"),
    (re.compile(r'\b(front[\s\-]?end\s+developer|frontend\s+developer)\b', re.IGNORECASE), "Frontend Developer"),
    (re.compile(r'\b(back[\s\-]?end\s+developer|backend\s+developer)\b', re.IGNORECASE), "Backend Developer"),
    (re.compile(r'\b(product\s+manager|product\s+owner)\b', re.IGNORECASE), "Product Manager"),
    (re.compile(r'\b(project\s+manager|scrum\s+master|delivery\s+manager|program\s+manager)\b', re.IGNORECASE), "Project Manager"),
    (re.compile(r'\b(business\s+analyst|functional\s+analyst|system\s+analyst)\b', re.IGNORECASE), "Business Analyst"),
    (re.compile(r'\b(hr\s+(manager|executive)|talent\s+acquisition|recruiter)\b', re.IGNORECASE), "Human Resources"),
    (re.compile(r'\b(sales\s+(manager|executive)|digital\s+marketing\s+(executive|manager)|seo\s+analyst)\b', re.IGNORECASE), "Sales / Marketing"),
    (re.compile(r'\b(financial\s+analyst|accountant|finance\s+manager)\b', re.IGNORECASE), "Finance / Accounting"),
    (re.compile(r'\b(network\s+engineer|network\s+administrator|system\s+administrator)\b', re.IGNORECASE), "Network Engineer"),
    (re.compile(r'\b(security\s+engineer|cybersecurity\s+engineer|penetration\s+tester|information\s+security)\b', re.IGNORECASE), "Cybersecurity Engineer"),
    (re.compile(r'\b(technical\s+writer|content\s+writer)\b', re.IGNORECASE), "Technical Writer"),
    (re.compile(r'\b(software\s+engineer|software\s+developer|application\s+developer|sde|swe)\b', re.IGNORECASE), "Software Developer"),
]

DOMAIN_GROUP_MAP = {
    ".NET Developer":"Software Development","Java Developer":"Software Development",
    "Python Developer":"Software Development","React Developer":"Software Development",
    "Angular Developer":"Software Development","Vue Developer":"Software Development",
    "Node.js Developer":"Software Development","Full Stack Developer":"Software Development",
    "Frontend Developer":"Software Development","Backend Developer":"Software Development",
    "Software Developer":"Software Development","Flutter Developer":"Mobile Development",
    "Android Developer":"Mobile Development","iOS Developer":"Mobile Development",
    "Data Scientist / ML Engineer":"Data Science / ML","Data Analyst":"Data Science / ML",
    "Data Engineer":"Data Science / ML","DevOps / Cloud Engineer":"DevOps / Cloud",
    "QA / Test Engineer":"QA / Testing","UI/UX Designer":"UI/UX Design",
    "Product Manager":"Product Management","Project Manager":"Project Management",
    "Business Analyst":"Business Analysis","Human Resources":"Human Resources",
    "Sales / Marketing":"Sales / Marketing","Finance / Accounting":"Finance / Accounting",
    "Network Engineer":"Network Engineering","Cybersecurity Engineer":"Cybersecurity",
    "Technical Writer":"Technical Writing","Unknown":"Unknown",
}

DOMAIN_LABELS = [
    "Software Development","QA / Testing","Data Science / ML","DevOps / Cloud",
    "UI/UX Design","Mobile Development","Product Management","Business Analysis",
    "Project Management","Human Resources","Sales / Marketing","Finance / Accounting",
    "Cybersecurity","Network Engineering","Technical Writing",
]

MODEL_CONFIDENCE_THRESHOLD = 0.55
SCORE_ZERO_THRESHOLD       = 35

# ── Section header patterns ──
_EXP_HEADER_RE = re.compile(
    r'^\s*(work\s+experience|professional\s+experience|employment\s+history|'
    r'experience|career\s+history|work\s+history|internship|'
    r'work\s+history\s+&\s+experience|employment\s+details?|'
    r'professional\s+background)\s*$',
    re.IGNORECASE
)
_EDU_HEADER_RE = re.compile(
    r'^\s*(education|academic|qualification|scholastic|'
    r'educational\s+background|academic\s+background|'
    r'academic\s+details?|educational\s+details?)\s*$',
    re.IGNORECASE
)
_NON_EXP_HEADER_RE = re.compile(
    r'^\s*(projects?|project\s+details?|academic\s+projects?|personal\s+projects?|'
    r'key\s+projects?|major\s+projects?|notable\s+projects?|skills?|'
    r'technical\s+skills?|core\s+skills?|key\s+skills?|certifications?|'
    r'achievements?|awards?|publications?|interests?|hobbies|extra.?curricular|'
    r'activities|education|academic|qualification|scholastic)\s*$',
    re.IGNORECASE
)

# ── Date patterns ──
_DATE_RANGE_RE = re.compile(
    r'(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|'
    r'jul(?:y)?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)'
    r'\s+\d{4}'
    r'|\d{4}\s*[-–]\s*(?:\d{4}|present|current|till\s*date)',
    re.IGNORECASE
)
_DATE_PAIR_RE = re.compile(
    r'((?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|'
    r'jul(?:y)?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)'
    r'\s+\d{4}|\d{4})'
    r'\s*(?:to|[-–])\s*'
    r'((?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|'
    r'jul(?:y)?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)'
    r'\s+\d{4}|\d{4}|present|current|till\s*date)',
    re.IGNORECASE
)

# ── Role words ──
_ROLE_WORDS_RE = re.compile(
    r'\b(engineer|developer|analyst|designer|manager|consultant|lead|head|'
    r'intern(?:ship)?|executive|specialist|architect|tester|scientist|associate|'
    r'coordinator|officer|director|programmer|technician|administrator|'
    r'devops|sde|swe|qa|ux|ui|frontend|backend|fullstack|trainee|apprentice)\b',
    re.IGNORECASE
)

# ── Metadata label lines ──
_METADATA_RE = re.compile(
    r'^(role|company|duration|location|responsibilities|period|'
    r'organisation|organization|employer|designation)\s*:\s*(.+)$',
    re.IGNORECASE
)

# ── Job title regex ──
JOB_TITLE_RE = re.compile(
    r'(?:^|\n)\s*'
    r'((?:senior|junior|jr\.?|sr\.?|lead|principal|associate|staff|assistant|trainee)?\s*'
    r'(?:'
    r'(?:qa|quality\s+assurance|quality\s+analyst|test(?:ing)?)\s*(?:engineer|analyst|lead|intern(?:ship)?)?|'
    r'software\s+test(?:er|ing)?\s*(?:intern(?:ship)?)?|'
    r'automation\s+tester|manual\s+tester|sdet|'
    r'(?:\.net|asp\.net|c#|csharp)\s+developer|'
    r'software\s+(?:engineer|developer)|'
    r'(?:frontend|front[\s\-]end|backend|back[\s\-]end|full[\s\-]?stack)\s+developer|'
    r'(?:mobile|android|ios|flutter|react\s+native)\s+developer|'
    r'data\s+(?:scientist|analyst|engineer)|'
    r'(?:machine\s+learning|ml|ai)\s+engineer|'
    r'(?:devops|cloud|platform|infrastructure|systems?)\s+engineer|'
    r'site\s+reliability\s+engineer|sre|'
    r'(?:ui[\s/]*ux|product|graphic|visual|interaction)\s+designer|'
    r'product\s+(?:manager|owner)|'
    r'business\s+analyst|project\s+manager|scrum\s+master|'
    r'(?:hr|human\s+resource)\s+(?:manager|executive)|recruiter|'
    r'(?:sales|marketing)\s+(?:manager|executive)|'
    r'(?:network|security|cybersecurity)\s+engineer|'
    r'technical\s+writer|financial\s+analyst|accountant|sde|swe'
    r'))',
    re.IGNORECASE | re.MULTILINE
)

FRESHER_KW_RE = re.compile(
    r'\b(fresher|fresh\s+graduate|no\s+experience|0\s+year|zero\s+experience|'
    r'recent\s+graduate|just\s+graduated|newly\s+graduated|'
    r'campus\s+hire|campus\s+placement)\b',
    re.IGNORECASE
)

MONTH_MAP = {
    "jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
    "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12,
    "january":1,"february":2,"march":3,"april":4,"june":6,
    "july":7,"august":8,"september":9,"october":10,
    "november":11,"december":12
}
MONTH_NAMES = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
               7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
_NAME_RE = re.compile(r'\b([A-Z][a-z]{1,20}(?:\s+[A-Z][a-z]{1,20}){1,3})\b')

# ── Education institution detector ──
_EDU_INSTITUTION_RE = re.compile(
    r'\b(university|college|institute\s+of\s+technology|iit|nit|bits|vit|'
    r'polytechnic|school\s+of|academy|institution|deemed|'
    r'b\.?tech|m\.?tech|b\.?e\.?|m\.?e\.?|bsc|msc|mba|bca|mca|'
    r'higher\s+secondary|matriculation|sslc|hsc|cbse|icse|'
    r'10th|12th|diploma|bachelor|master|phd|doctorate)\b',
    re.IGNORECASE
)
# Degree keyword (used to detect education date context)
_DEGREE_KW_RE = re.compile(
    r'\b(b\.?tech|m\.?tech|b\.?e|m\.?e|bsc|msc|mba|bca|mca|'
    r'bachelor|master|diploma|degree|10th|12th|sslc|hsc|phd|'
    r'university|college|institute)\b',
    re.IGNORECASE
)

In [5]:
# Line Helpers


def _is_date_line(line):
    return bool(_DATE_RANGE_RE.search(line.strip()))

def _is_bullet(line):
    s = line.strip()
    if not s: return False
    if s[0] in ('-','•','*','·','–','▪','○','●'): return True
    if len(s) > 110: return True
    return False

def _parse_meta(line):
    m = _METADATA_RE.match(line.strip())
    if m: return m.group(1).lower(), m.group(2).strip()
    return None, None

def _is_meta(line):
    return _METADATA_RE.match(line.strip()) is not None

def _is_edu_line(line):
    """True if this line is clearly an education institution or degree line."""
    return bool(_EDU_INSTITUTION_RE.search(line))

def _is_company_line(line):
    """
    True if line looks like a company name.
    - 3–70 chars, starts with capital
    - Not a date, bullet, metadata, or education institution line
    - Does not contain role words
    - At most 8 words, at most 2 lowercase-starting words
    """
    s = line.strip()
    if not s or len(s) < 3 or len(s) > 70: return False
    if _is_date_line(s): return False
    if _is_bullet(s): return False
    if _is_meta(s): return False
    if _is_edu_line(s): return False
    if not re.match(r'^[A-Z]', s): return False
    words = s.split()
    if len(words) > 8: return False
    if sum(1 for w in words if w and w[0].islower()) > 2: return False
    if _ROLE_WORDS_RE.search(s): return False
    return True

In [6]:
# Section Boundary Detection & Experience Line Extraction


def _find_section_boundaries(lines):
    boundaries = []
    for i, line in enumerate(lines):
        if _EXP_HEADER_RE.match(line):
            boundaries.append((i, 'experience'))
        elif _EDU_HEADER_RE.match(line):
            boundaries.append((i, 'education'))
        elif _NON_EXP_HEADER_RE.match(line):
            boundaries.append((i, 'other'))
    return boundaries


def get_experience_lines(full_text):
    """
    Returns ONLY lines that belong to the experience section(s).

    Strategy:
    1. Find all section headers → collect lines between exp header and next
       non-exp header.
    2. If no section headers found → use top 65% of resume, but stop at
       the FIRST line that is clearly an education entry (not just a mention
       of degree in summary — must be a standalone short edu line).

    IMPORTANT: This function must NOT stop early just because the word
    "B.Tech" or "degree" appears in the candidate's objective/summary line.
    We only stop if the line is SOLELY an education entry (short, standalone).
    """
    lines      = full_text.splitlines()
    n          = len(lines)
    boundaries = _find_section_boundaries(lines)

    if boundaries:
        exp_lines = []
        in_exp    = False
        for i, line in enumerate(lines):
            btype = next((bt for bi, bt in boundaries if bi == i), None)
            if btype == 'experience':
                in_exp = True
                continue
            elif btype in ('education', 'other') and in_exp:
                in_exp = False
                continue
            elif btype in ('education', 'other'):
                continue
            if in_exp:
                exp_lines.append(line)
        if exp_lines:
            return exp_lines


    cutoff = int(n * 0.65)
    result = []
    for i, line in enumerate(lines[:cutoff]):
        s = line.strip()
        # Stop if explicit education header
        if _EDU_HEADER_RE.match(s):
            break

        if (len(s) < 80
                and s
                and not _is_bullet(s)
                and _DEGREE_KW_RE.search(s)
                and not _ROLE_WORDS_RE.search(s)
                and not _is_date_line(s)):
            # Extra guard: skip if it has role context in surrounding lines
            # (means it's part of a job description, not an edu section)
            surrounding = " ".join(lines[max(0,i-2):i+3]).lower()
            if not any(w in surrounding for w in ("engineer","developer","analyst",
                                                    "tester","designer","manager")):
                break
        result.append(line)
    return result


def get_education_lines(full_text):
    lines      = full_text.splitlines()
    boundaries = _find_section_boundaries(lines)
    edu_lines  = []
    in_edu     = False
    for i, line in enumerate(lines):
        btype = next((bt for bi, bt in boundaries if bi == i), None)
        if btype == 'education':
            in_edu = True; continue
        elif btype in ('experience', 'other') and in_edu:
            in_edu = False; continue
        if in_edu:
            edu_lines.append(line)
    return edu_lines

In [7]:
#Fresher Detection


def is_fresher(full_text):
    """
    Experienced if there is at least one complete date-pair in experience lines.
    Fresher if: no date-pairs in experience lines, or explicit fresher keyword.
    """
    exp_lines = get_experience_lines(full_text)
    exp_text  = "\n".join(exp_lines)
    pairs     = _DATE_PAIR_RE.findall(exp_text)
    if pairs:
        return False
    if FRESHER_KW_RE.search(full_text):
        return True
    return True

In [8]:
#  Name Extraction

def get_name_from_filename(fp):
    raw = os.path.splitext(os.path.basename(fp))[0]
    raw = re.sub(r'\[.*?\]|\(.*?\)', ' ', raw)
    raw = re.sub(r'([a-z])([A-Z])', r'\1 \2', raw)
    raw = re.sub(r'[_\-\.\s]+', ' ', raw).strip()
    good = [t.capitalize() for t in raw.split()
            if t and t.lower() not in INVALID_WORDS
            and not any(c.isdigit() for c in t)
            and not re.search(r'[^A-Za-z]', t) and len(t) >= 2]
    return " ".join(good) if good else None

def get_name_bert(text, model):
    try:
        ents = model(text[:700])
        names = [e['word'] for e in ents if e['entity_group'] == 'PER']
        if names:
            raw = " ".join(names[:4]).replace(" ##","").strip()
            parts = [w for w in raw.split()
                     if w.lower() not in INVALID_WORDS
                     and not any(c.isdigit() for c in w) and len(w) >= 2]
            if len(parts) >= 2: return " ".join(parts[:4])
    except: pass
    return None

def get_name_spacy(text):
    try:
        doc = nlp(text[:700])
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                c = ent.text.strip(); ws = c.split()
                if (2 <= len(ws) <= 5
                        and all(not any(d.isdigit() for d in w) for w in ws)
                        and all(w.lower() not in INVALID_WORDS for w in ws)):
                    return c
    except: pass
    return None

def get_name_regex(text):
    top = "\n".join(text.splitlines()[:12])
    for m in _NAME_RE.findall(top):
        if all(w.lower() not in INVALID_WORDS for w in m.split()):
            return m
    return None

def extract_name(full_text, left_col, right_col, filepath):
    r = get_name_from_filename(filepath)
    if r: return r, "filename"
    top = "\n".join(full_text.splitlines()[:10])
    r = get_name_bert(top, ner_pipe);
    if r: return r, "bert-base-top"
    r = get_name_bert(top, ner_large);
    if r: return r, "bert-large-top"
    r = get_name_spacy(top);
    if r: return r, "spacy-top"
    r = get_name_bert(full_text, ner_pipe);
    if r: return r, "bert-base-full"
    r = get_name_bert(full_text, ner_large);
    if r: return r, "bert-large-full"
    r = get_name_spacy(full_text);
    if r: return r, "spacy-full"
    r = get_name_regex(full_text);
    if r: return r, "regex"
    return "Name Not Found", "none"

In [9]:
# Career Break Detection

def _parse_date_token(token):
    t = token.strip().lower()
    if t in ("present","current","till date","now"):
        now = datetime.now(); return now.month, now.year
    m = re.match(r'([a-z]+)\s+(\d{4})', t)
    if m:
        mon = MONTH_MAP.get(m.group(1)); yr = int(m.group(2))
        if mon: return mon, yr
    m = re.match(r'(\d{4})$', t)
    if m: return 6, int(m.group(1))
    return None

def _extract_date_ranges(text):
    ranges = []
    for match in _DATE_PAIR_RE.finditer(text):
        s = _parse_date_token(match.group(1))
        e = _parse_date_token(match.group(2))
        if s and e:
            sm = s[1]*12+s[0]; em = e[1]*12+e[0]
            if em >= sm: ranges.append((sm, em))
    return sorted(ranges)

def _abs_to_my(abs_m):
    yr = (abs_m-1)//12; mon = abs_m - yr*12; return mon, yr

def detect_career_break(full_text):
    exp_text = "\n".join(get_experience_lines(full_text))
    ranges   = _extract_date_ranges(exp_text)
    if len(ranges) < 2: return False, "No career break detected"
    breaks = []
    for i in range(1, len(ranges)):
        gap = ranges[i][0] - ranges[i-1][1]
        if gap >= 6:
            bsm, bsy = _abs_to_my(ranges[i-1][1])
            bem, bey = _abs_to_my(ranges[i][0])
            sl = f"{MONTH_NAMES.get(bsm,'?')} {bsy}"
            el = f"{MONTH_NAMES.get(bem,'?')} {bey}"
            yrs = gap//12; mons = gap%12
            if yrs > 0 and mons > 0: dur = f"{yrs}yr {mons}mo"
            elif yrs > 0:             dur = f"{yrs}yr{'s' if yrs>1 else ''}"
            else:                     dur = f"{mons}mo{'s' if mons>1 else ''}"
            breaks.append(f"{sl} -> {el} ({dur})")
    if breaks: return True, "Career break: " + "  |  ".join(breaks)
    return False, "No career break detected"

In [10]:
# ════════════════════════════════════════════════════════════════════════
#  CELL 10 — Domain Detection
# ════════════════════════════════════════════════════════════════════════
def _domain_from_title(text):
    if not text: return None
    for pat, label in SPECIFIC_TECH_DOMAIN_MAP:
        if pat.search(text): return label
    return None

def _domain_from_model(text):
    try:
        r = classifier(text[:600], DOMAIN_LABELS, multi_label=False)
        if r["scores"][0] >= MODEL_CONFIDENCE_THRESHOLD:
            return r["labels"][0], r["scores"][0]
    except: pass
    return None, 0.0

def _resolve_block_domain(block_text):
    m = JOB_TITLE_RE.search(block_text)
    if m:
        d = _domain_from_title(m.group(1).strip())
        if d: return d
    for line in block_text.splitlines()[:12]:
        s = line.strip()
        if not s or _is_bullet(s) or len(s) > 100: continue
        lt, val = _parse_meta(s)
        if lt in ('role','designation') and val:
            d = _domain_from_title(val)
            if d: return d
            continue
        if _is_meta(s): continue
        d = _domain_from_title(s)
        if d: return d
    cat, _ = _domain_from_model(block_text)
    return cat or "Unknown"

def _split_into_job_blocks(exp_lines):
    n    = len(exp_lines)
    tags = []
    for line in exp_lines:
        s = line.strip()
        lt, val = _parse_meta(s)
        if lt == 'duration' or _is_date_line(s):
            tags.append('date')
        elif lt in ('company','organisation','organization','employer'):
            tags.append('company')
        elif lt in ('role','designation') and val:
            tags.append('role')
        elif _is_bullet(s):
            tags.append('bullet')
        elif _ROLE_WORDS_RE.search(s) and 3 < len(s) < 100:
            tags.append('role')
        elif _is_company_line(s):
            tags.append('company')
        else:
            tags.append('other')

    split_pts = set()
    for i, tag in enumerate(tags):
        if tag != 'date': continue
        found = False
        for j in range(max(0, i-5), i):
            if tags[j] in ('company', 'role'):
                split_pts.add(j); found = True; break
        if not found:
            for j in range(i+1, min(n, i+4)):
                if tags[j] in ('company', 'role'):
                    split_pts.add(i); break

    split_pts = sorted(split_pts)
    if not split_pts:
        blocks, cur = [], []
        for i, line in enumerate(exp_lines):
            if tags[i] == 'date' and cur:
                blocks.append(cur); cur = [line]
            else:
                cur.append(line)
        if cur: blocks.append(cur)
        return ["\n".join(b) for b in blocks if any(s.strip() for s in b)]

    blocks, prev = [], 0
    for sp in split_pts:
        if sp > prev:
            b = "\n".join(exp_lines[prev:sp])
            if b.strip(): blocks.append(b)
        prev = sp
    last = "\n".join(exp_lines[prev:])
    if last.strip(): blocks.append(last)
    return blocks

def detect_domains(full_text, fresher):
    if fresher:
        m = JOB_TITLE_RE.search(full_text)
        d = _domain_from_title(m.group(1).strip()) if m else None
        if not d: d, _ = _domain_from_model(full_text[:1200])
        return d or "Unknown", None, False

    exp_lines = get_experience_lines(full_text)
    if not exp_lines:
        d, _ = _domain_from_model(full_text[:1500])
        return d or "Unknown", None, False

    blocks = _split_into_job_blocks(exp_lines)
    if not blocks:
        d = _resolve_block_domain("\n".join(exp_lines[:50]))
        return d or "Unknown", None, False

    block_domains = []
    for block in blocks:
        cat = _resolve_block_domain(block)
        if cat and cat != "Unknown":
            block_domains.append(cat)

    if not block_domains:
        return "Unknown", None, False

    deduped = []
    for cat in block_domains:
        if not deduped or deduped[-1] != cat:
            deduped.append(cat)

    current     = deduped[-1]
    current_grp = DOMAIN_GROUP_MAP.get(current, current)
    prev_domain = None
    for cat in deduped[:-1]:
        if DOMAIN_GROUP_MAP.get(cat, cat) != current_grp:
            prev_domain = cat

    if prev_domain:
        return current, prev_domain, True
    return current, None, False

In [11]:
#Location Extraction

# FIX: Removed lazy ? from quantifiers. Now greedy → captures full city name.
_LOC_META_RE = re.compile(
    r'^\s*location\s*:\s*([A-Za-z][A-Za-z\s\-\.]{1,40})',
    re.IGNORECASE
)

def _city_match(raw):

    if not raw: return None
    # Step 1: clean
    raw = raw.strip()
    # Step 2: cut at comma (removes country suffix)
    raw = raw.split(',')[0].strip()
    # Step 3: cut at slash
    raw = raw.split('/')[0].strip()
    raw_lower = raw.lower()

    # Step 4: exact match
    if raw_lower in KNOWN_CITIES:
        return raw_lower.title()

    # Step 5: word-boundary match (longest city first to prefer "new delhi" over "delhi")
    for city in sorted(KNOWN_CITIES, key=len, reverse=True):
        if len(city) < 3: continue
        if re.search(r'\b' + re.escape(city) + r'\b', raw_lower):
            return city.title()

    return None


def extract_company_locations(full_text, left_col=None, right_col=None):

    exp_lines = get_experience_lines(full_text)
    if not exp_lines:
        return "Not mentioned"

    candidates = []

    for line in exp_lines:
        s = line.strip()
        if not s or _is_bullet(s):
            continue

        # ── Source 1: "Location: <city>" metadata label ──
        loc_m = _LOC_META_RE.match(s)
        if loc_m:
            # Group(1) is now greedy — captures "Trivandrum, India" or "Trivandrum"
            # _city_match() handles splitting at comma to drop country
            city = _city_match(loc_m.group(1))
            if city:
                candidates.append(city)
            continue  # don't try other sources on this line

        # ── Source 2: "CompanyName, City" or "CompanyName | City" ──
        if not _is_date_line(s) and not _is_meta(s):
            # Check for comma or pipe separator
            for sep in (',', '|'):
                if sep in s:
                    parts = s.split(sep, 1)
                    # Try city from the part AFTER the separator
                    city = _city_match(parts[1].strip())
                    if city:
                        candidates.append(city)
                        break
                    # Also try the part BEFORE (e.g. "Bangalore | TCS" format)
                    city = _city_match(parts[0].strip())
                    if city:
                        candidates.append(city)
                        break

        # ── Source 3: Standalone city-only line ──
        # The line must be ONLY a city name (or city + work-mode like "Kochi, Remote")
        if (not _is_date_line(s)
                and not _is_meta(s)
                and not _ROLE_WORDS_RE.search(s)
                and not _EDU_INSTITUTION_RE.search(s)
                and len(s) < 40):
            city = _city_match(s)
            # Accept if the ENTIRE line (after cleaning) is just the city name
            if city and s.lower().strip() in (city.lower(), city.lower() + "."):
                candidates.append(city)

    # ── Source 4: NER on all experience lines ──
    exp_text = "\n".join(exp_lines)
    try:
        # Run NER in chunks to handle long experience sections
        chunk_size = 800
        for start in range(0, min(len(exp_text), 4000), chunk_size):
            chunk = exp_text[start:start + chunk_size]
            for e in ner_pipe(chunk):
                if e['entity_group'] in ('LOC', 'GPE'):
                    word = e['word'].replace(" ##","").strip()
                    city = _city_match(word)
                    if city:
                        candidates.append(city)
    except:
        pass

    # ── Deduplicate & validate ──
    seen, unique = set(), []
    for loc in candidates:
        if not loc: continue
        ll = loc.strip().lower()
        if ll in KNOWN_CITIES and ll not in seen:
            seen.add(ll)
            unique.append(loc.strip().title())

    return ", ".join(unique) if unique else "Not mentioned"


In [12]:
# Company Count


def count_companies(full_text, fresher=False):
    if fresher: return 0

    exp_lines = get_experience_lines(full_text)
    if not exp_lines: return 0

    # Strategy A: explicit "Company: <name>" metadata labels
    company_labels = set()
    for line in exp_lines:
        lt, val = _parse_meta(line)
        if lt in ('company','organisation','organization','employer') and val:
            if not _is_edu_line(val):
                company_labels.add(val.strip().lower())
    if company_labels:
        return len(company_labels)

    # Strategy B: structural anchor detection
    n    = len(exp_lines)
    tags = []
    for line in exp_lines:
        s = line.strip()
        lt, val = _parse_meta(s)
        if lt == 'duration' or _is_date_line(s):
            tags.append('date')
        elif lt in ('company','organisation','organization','employer'):
            tags.append('company')
        elif lt in ('role','designation') and val:
            tags.append('role')
        elif _is_bullet(s):
            tags.append('bullet')
        elif _ROLE_WORDS_RE.search(s) and 3 < len(s) < 100 and not _is_edu_line(s):
            tags.append('role')
        elif _is_company_line(s):
            tags.append('company')
        else:
            tags.append('other')

    anchor_set = set()
    for i, tag in enumerate(tags):
        if tag != 'date': continue
        found = False
        for j in range(max(0, i-5), i):
            if tags[j] in ('company', 'role'):
                anchor_set.add(j); found = True; break
        if not found:
            for j in range(i+1, min(n, i+4)):
                if tags[j] in ('company', 'role'):
                    anchor_set.add(i); break

    count = len(anchor_set)

    if count == 0:
        date_pos = [i for i,t in enumerate(tags) if t == 'date']
        if date_pos:
            clusters = 1
            for k in range(1, len(date_pos)):
                if date_pos[k] - date_pos[k-1] > 8:
                    clusters += 1
            count = clusters

    return count


In [13]:
# Match Score

def compute_match_score(jd_text, resume_text):
    je  = embedder.encode(jd_text,            convert_to_tensor=True)
    re_ = embedder.encode(resume_text[:3000], convert_to_tensor=True)
    raw = round(util.cos_sim(je, re_).item() * 100, 2)
    return 0.0 if raw < SCORE_ZERO_THRESHOLD else raw


In [14]:
# Build Summary

def build_summary(name, method, current_d, prev_d, switched,
                  has_break, break_desc, location_str, company_count, fresher):
    rows = [
        f"Candidate      : {name}  (via {method})",
        f"Candidate Type : {'Fresher' if fresher else 'Experienced'}",
        f"Current Domain : {current_d}",
    ]
    if fresher:
        rows += ["Domain Switch  : Not applicable (Fresher)",
                 "Career Break   : No Career Break"]
    else:
        rows.append(f"Domain Switch  : {'Yes  (' + prev_d + '  ->  ' + current_d + ')  [!]' if switched else 'No'}")
        rows.append(f"Career Break   : {'[!] ' + break_desc if has_break else 'No career break detected'}")
    rows.append(f"Work Locations : {location_str}")
    rows.append(f"Companies      : {company_count} {'company' if company_count==1 else 'companies'}")
    return "\n".join(rows)


In [15]:
# analyze_resume

def analyze_resume(jd_text, path, threshold=70, original_filename=None):
    fname                          = os.path.basename(original_filename or path)
    full_text, left_col, right_col = extract_text(path)

    if not full_text.strip():
        return {"file":fname,"score":0,"status":"ERROR","name":"N/A",
                "candidate_type":"-","domain":"-","domain_switch":False,
                "domain_switch_label":"No","domain_switch_from":"",
                "career_break":False,"career_break_detail":"-",
                "locations":"Not mentioned","company_count":0,
                "summary":"Could not extract text from file."}

    name, method  = extract_name(full_text, left_col, right_col, original_filename or path)
    score         = compute_match_score(jd_text, full_text)
    status        = "MATCHED" if score >= threshold else "NOT MATCHED"
    fresher       = is_fresher(full_text)

    current_d, prev_d, switched = detect_domains(full_text, fresher)

    if fresher:
        has_break, break_desc = False, "No Career Break"
    else:
        has_break, break_desc = detect_career_break(full_text)

    location_str  = extract_company_locations(full_text)
    company_count = count_companies(full_text, fresher=fresher)
    switch_label  = ("Not applicable (Fresher)" if fresher else
                     (f"Yes  ({prev_d}  ->  {current_d})" if switched else "No"))

    summary = build_summary(name, method, current_d, prev_d, switched,
                            has_break, break_desc, location_str, company_count, fresher)
    return {
        "file"               : fname,
        "score"              : score,
        "status"             : status,
        "name"               : name,
        "candidate_type"     : "Fresher" if fresher else "Experienced",
        "domain"             : current_d,
        "domain_switch"      : switched if not fresher else False,
        "domain_switch_label": switch_label,
        "domain_switch_from" : prev_d if (switched and not fresher) else "",
        "career_break"       : has_break,
        "career_break_detail": break_desc,
        "locations"          : location_str,
        "company_count"      : company_count,
        "summary"            : summary
    }


In [16]:
# Display Results

def display_results(results, threshold):
    clear_output(wait=True)
    print(f"\n{'='*75}")
    print(f"   RESUME MATCHING REPORT  |  Threshold: {threshold}%")
    print(f"{'='*75}\n")
    matched = sum(1 for r in results if r.get("status") == "MATCHED")
    print(f"Total: {len(results)}  |  Matched: {matched}  |  Not Matched: {len(results)-matched}\n")
    print("-"*75)

    for r in sorted(results, key=lambda x: x.get("score",0), reverse=True):
        print(f"\n  File        : {r['file']}")
        print(f"  Match Score : {r['score']}%  ->  {r['status']}")
        print("  -- Summary -----------------------------------------------")
        for line in r["summary"].splitlines():
            print(f"  {line}")
        print("-"*75)

    rows = []
    for r in results:
        nl = next((l for l in r["summary"].splitlines() if "Candidate" in l and "via" in l), "")
        nv = nl.split(":",1)[-1].split("(via")[0].strip() if nl else "-"
        rows.append({
            "File"          : r["file"],
            "Name"          : nv,
            "Type"          : r.get("candidate_type","-"),
            "Score (%)"     : r["score"],
            "Status"        : r["status"],
            "Domain"        : r.get("domain","-"),
            "Domain Switch" : r.get("domain_switch_label","No"),
            "Career Break"  : r.get("career_break_detail","-"),
            "Work Locations": r.get("locations","Not mentioned"),
            "Companies"     : r.get("company_count",0),
            "Summary"       : r["summary"].replace("\n"," | ")
        })

    df = pd.DataFrame(rows)
    print("\n SUMMARY TABLE:\n")
    display(df.sort_values("Score (%)", ascending=False).reset_index(drop=True))

In [17]:
#colab Interactive Main

print("\n" + "="*75)
print("  RESUME MATCHER -- Match Score + Consolidated Summary")
print("="*75)
print("\n STEP 1: Paste Job Description. Type END when done:\n")
jd_lines = []
while True:
    line = input()
    if line.strip().upper() == "END": break
    jd_lines.append(line)
jd_text = "\n".join(jd_lines)
print(f" JD captured -- {len(jd_text)} chars")
t = input("\n STEP 2: Threshold % (Enter = 70): ").strip()
threshold = int(t) if t.isdigit() else 70
print(f" Threshold: {threshold}%")
print("\n STEP 3: Upload resumes (PDF / DOCX / TXT):")
uploaded = files.upload()
resume_paths = []
for fname, fdata in uploaded.items():
    p = f"/content/{fname}"
    with open(p,"wb") as f: f.write(fdata)
    resume_paths.append(p)
print(f"\n {len(resume_paths)} file(s) uploaded. Analysing...\n")
results = []
for i, path in enumerate(resume_paths):
    print(f"  [{i+1}/{len(resume_paths)}] {os.path.basename(path)}")
    results.append(analyze_resume(jd_text, path, threshold))
display_results(results, threshold)


   RESUME MATCHING REPORT  |  Threshold: 50%

Total: 1  |  Matched: 1  |  Not Matched: 0

---------------------------------------------------------------------------

  File        : Naukri_Yogeshrajaa[3y_0m].pdf
  Match Score : 55.38%  ->  MATCHED
  -- Summary -----------------------------------------------
  Candidate      : Yogeshrajaa  (via filename)
  Candidate Type : Experienced
  Current Domain : QA / Testing
  Domain Switch  : No
  Career Break   : No career break detected
  Work Locations : Not mentioned
  Companies      : 2 companies
---------------------------------------------------------------------------

 SUMMARY TABLE:



,File,Name,Type,Score (%),Status,Domain,Domain Switch,Career Break,Work Locations,Companies,Summary
0,Naukri_Yogeshrajaa[3y_0m].pdf,Yogeshrajaa,Experienced,55.38,MATCHED,QA / Testing,No,No career break detected,Not mentioned,2,Candidate : Yogeshrajaa (via filename) |...


In [18]:

#  Flask API

app = Flask(__name__)

@app.route("/api/health", methods=["GET"])
def health():
    return jsonify({"status":"ok","message":"Resume Matcher API is running"})

@app.route("/api/analyze-batch", methods=["POST"])
def analyze_batch():
    ufiles = request.files.getlist("files")
    jd     = request.form.get("jd_text","")
    th     = int(request.form.get("threshold",70))
    if not ufiles: return jsonify({"error":"No files uploaded"}), 400
    if not jd.strip(): return jsonify({"error":"jd_text is required"}), 400

    results = []
    for rf in ufiles:
        safe = os.path.basename((rf.filename or "").replace("\\","/")).strip() or "unknown.pdf"
        suf  = os.path.splitext(safe)[-1] or ".pdf"
        with tempfile.NamedTemporaryFile(delete=False, suffix=suf) as tmp:
            rf.save(tmp.name); tp = tmp.name
        try:
            res = analyze_resume(jd, tp, th, original_filename=safe)
            res["file"] = safe; results.append(res)
        except Exception as e:
            results.append({"file":safe,"error":str(e)})
        finally:
            os.unlink(tp)

    matched = sum(1 for r in results if r.get("status")=="MATCHED")
    return jsonify({"total":len(results),"matched":matched,
                    "not_matched":len(results)-matched,"threshold":th,
                    "results":sorted(results,key=lambda x:x.get("score",0),reverse=True)})

@app.route("/api/analyze", methods=["POST"])
def analyze_single():
    if "file" not in request.files: return jsonify({"error":"No file uploaded"}),400
    if not request.form.get("jd_text","").strip(): return jsonify({"error":"jd_text required"}),400
    rf   = request.files["file"]
    jd   = request.form["jd_text"]
    th   = int(request.form.get("threshold",70))
    safe = os.path.basename((rf.filename or "").replace("\\","/")).strip() or "unknown.pdf"
    suf  = os.path.splitext(safe)[-1] or ".pdf"
    with tempfile.NamedTemporaryFile(delete=False,suffix=suf) as tmp:
        rf.save(tmp.name); tp=tmp.name
    try:
        res = analyze_resume(jd,tp,th,original_filename=safe)
        res["file"] = safe; return jsonify(res)
    except Exception as e:
        return jsonify({"error":str(e)}),500
    finally:
        os.unlink(tp)

In [19]:
!lsof -i:5000

In [20]:
!kill -9 7047

/bin/bash: line 1: kill: (7047) - No such process


In [21]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess, time, re, threading

def run_flask():
    app.run(port=5000)

threading.Thread(target=run_flask, daemon=True).start()
time.sleep(2)

proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stderr=subprocess.PIPE
)

for line in proc.stderr:
    line = line.decode()
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print(f"\nPublic URL: {public_url}")t
        print(f"Use this in appsettings.json:")
        print(f'  "PythonApiBaseUrl": "{public_url}"')
        break

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit



Public URL: https://multi-suppliers-responding-alignment.trycloudflare.com
Use this in appsettings.json:
  "PythonApiBaseUrl": "https://multi-suppliers-responding-alignment.trycloudflare.com"
